# Nifty Options: From EDA to Model-Ready Data

This notebook completes preprocessing of the Nifty options dataset already in this workspace. It reads the full-data EDA, applies documented inclusion rules, builds causal features, and loads the chronological train/validation/test arrays. **No model is trained here.** The last section provides answers to the assignment questions that can be copied directly.

The chosen task is **next-minute option-price-change forecasting**: predict `log(close[t+1] / close[t])` using information available after bar `t` closes. A predicted log return can be converted into a predicted next close using `close[t] * exp(predicted_log_return)`. A theoretical option-pricing experiment is a different task and still needs verified expiry/contract identity and synchronized underlying prices, plus rates/dividends or other inputs required by its pricing specification. The data contain historical OHLC fields rather than verified executable bid/ask quotes.

This extends `01_nifty_eda.ipynb`. Raw files remain unchanged. Run this notebook using the repository's `.conda` Python kernel. It reads cached preparation results by default; set `REBUILD = True` below to regenerate them from the current inputs.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import Markdown, display

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'scripts/nifty_preprocess.py').exists()
)
EDA = ROOT / 'reports/nifty_eda'
REPORT = ROOT / 'reports/nifty_preprocessing'
DATA = ROOT / 'data/processed/nifty_training'
REBUILD = False

required_outputs = [
    REPORT / 'summary.json', REPORT / 'split_summary.csv',
    REPORT / 'exclusions.csv', REPORT / 'date_splits.csv',
    DATA / 'feature_spec.json', DATA / 'scaler.json', DATA / 'manifest.json',
    *[DATA / split / filename for split in ('train', 'valid', 'test')
      for filename in ('X.npy', 'y.npy', 'metadata.parquet')],
]
if REBUILD or not all(path.exists() for path in required_outputs):
    subprocess.run([sys.executable, 'scripts/nifty_preprocess.py'], cwd=ROOT, check=True)

eda = json.loads((EDA / 'summary.json').read_text())
descriptive = pd.read_csv(EDA / 'tables/describe.csv')
all_descriptive = descriptive.loc[descriptive['group'].eq('all')].set_index('metric')
close_stats = all_descriptive.loc['close']
volume_stats = all_descriptive.loc['volume']
summary = json.loads((REPORT / 'summary.json').read_text())
feature_spec = json.loads((DATA / 'feature_spec.json').read_text())
scaler = json.loads((DATA / 'scaler.json').read_text())
manifest = json.loads((DATA / 'manifest.json').read_text())
splits = pd.read_csv(REPORT / 'split_summary.csv')
exclusions = pd.read_csv(REPORT / 'exclusions.csv')
date_splits = pd.read_csv(REPORT / 'date_splits.csv')
feature_columns = feature_spec['feature_columns']
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')

print('Repository:', ROOT)
print('Preparation report:', REPORT / 'summary.json')
print('Local training arrays:', DATA)
print('Target:', summary['target'])
print('Prepared at (UTC):', manifest['created_at_utc'])
assert manifest['status'] == 'complete'

Repository: /Users/kuijun_2/Desktop/fall-2026-AI-financial-models
Preparation report: /Users/kuijun_2/Desktop/fall-2026-AI-financial-models/reports/nifty_preprocessing/summary.json
Local training arrays: /Users/kuijun_2/Desktop/fall-2026-AI-financial-models/data/processed/nifty_training
Target: target_log_return = log(close[t+1] / close[t]); exactly one minute ahead
Prepared at (UTC): 2026-10-14T17:56:55.342294+00:00


## 1. Dataset and EDA findings

The original data are headerless CSV/TXT files containing symbols, dates, times, OHLC prices, volume, and sometimes a ninth field. The existing inventory selects one representation of each archived source file after comparing file hashes, preventing identical CSV/TXT copies from doubling the sample. Archive folders are provenance; their dates are not assumed to be expiries or observation dates.

The audit below uses **all selected observations**, not just a sample. Numeric price/volume fields are continuous or count data; call/put is categorical; the timestamp gives temporal order. Data quality, skew, missing fields, coverage, and temporal dependence are therefore the relevant EDA topics. The earlier notebook contains the distribution and coverage plots.

In [2]:
inventory = eda['inventory_summary']
overview = pd.DataFrame([
    ('Raw CSV/TXT files in the workspace', inventory['physical_files']),
    ('Files selected after identical-representation removal', eda['selected_files']),
    ('Selected observations before row cleaning', eda['selected_rows']),
    ('Duplicate representation rows excluded', inventory['excluded_identical_representation_rows']),
    ('Observed dates before row cleaning', eda['observed_dates']),
], columns=['Metric', 'Count'])
display(overview)
print('Observed timestamps:', eda['first_timestamp'], 'through', eda['last_timestamp'])

quality_keys = {
    'ohlc_inconsistent': 'Inconsistent OHLC values',
    'outside_reference_session': 'Outside 09:15–15:29 reference window',
    'zero_volume': 'Zero reported volume',
    'close_at_most_1': 'Close price at most 1',
    'flat_ohlc': 'Open = high = low = close',
    'missing_extra_field_9': 'Missing unverified ninth field',
}
quality = pd.DataFrame([
    {'Finding': label, 'Rows': eda['quality_counts'][key],
     'Percent of selected rows': 100 * eda['quality_counts'][key] / eda['selected_rows']}
    for key, label in quality_keys.items()
])
display(quality)
print('These EDA flags may overlap; their counts must not be added as exclusions.')
display(all_descriptive.loc[['close', 'volume'], ['count', 'min', 'median', 'mean', 'p99', 'max']])
display(Markdown(
    f"**Distribution insight:** close price has median {close_stats['median']:,.2f} "
    f"and mean {close_stats['mean']:,.2f}; reported volume has median "
    f"{volume_stats['median']:,.0f} and mean {volume_stats['mean']:,.2f}. "
    'The means exceed the medians and the upper tails are long, consistent with right-skewed '
    'distributions. Log prices and log1p volumes compress these scales without deleting '
    'valid high values. These are mixed-contract summaries, not price trends or evidence '
    'about trading profitability; volume units remain unverified.'
))

Observed timestamps: 2016-12-14 09:15:00 through 2020-12-31 15:29:00
These EDA flags may overlap; their counts must not be added as exclusions.


,Metric,Count
0,Raw CSV/TXT files in the workspace,5682
1,Files selected after identical-representation ...,3137
2,Selected observations before row cleaning,17611169
3,Duplicate representation rows excluded,12668930
4,Observed dates before row cleaning,1004


,Finding,Rows,Percent of selected rows
0,Inconsistent OHLC values,2952,0.016762
1,Outside 09:15–15:29 reference window,34662,0.196818
2,Zero reported volume,396423,2.250975
3,Close price at most 1,491095,2.788543
4,Open = high = low = close,3347179,19.006001
5,Missing unverified ninth field,14979866,85.058896


,count,min,median,mean,p99,max
metric,,,,,,
close,17611169,0.050000,88.000000,171.707074,"1,250.000000","5,951.000000"
volume,17611169,0.000000,"1,125.000000","9,228.147298","130,050.000000","7,772,100.000000"


**Distribution insight:** close price has median 88.00 and mean 171.71; reported volume has median 1,125 and mean 9,228.15. The means exceed the medians and the upper tails are long, consistent with right-skewed distributions. Log prices and log1p volumes compress these scales without deleting valid high values. These are mixed-contract summaries, not price trends or evidence about trading profitability; volume units remain unverified.

The EDA found incomplete minute coverage: the median source-file/day has only 219 of the 375 reference-window minutes. Consecutive-minute examples must therefore be constructed explicitly. Filling every missing minute would invent observations and change the meaning of the target.

Zero-volume records, small positive option prices, and flat bars are retained with indicators because those values are not sufficient evidence of a data error. Extreme observed returns are also retained; arbitrary clipping could erase the behavior a forecasting model needs to evaluate. These groups warrant separate evaluation when modeling begins. The unknown ninth field has structural missingness and unverified semantics, so it is excluded from model features rather than imputed or labeled as open interest.

## 2. Inclusion and exclusion rules

| Issue | Preparation rule | Reason |
|---|---|---|
| Identical CSV/TXT representations | Use the inventory's selected representation | Avoid duplicated observations and samples |
| Invalid timestamps, required numeric values, or symbol parsing | Exclude unusable rows | Cannot define reliable features/labels |
| Nonpositive OHLC, negative volume, inconsistent OHLC | Exclude invalid rows | Log features require positive prices and coherent bars |
| Repeated source/timestamp records | Remove exact repeated rows; quarantine conflicting values | Avoid ambiguous source-series observations |
| Timestamps with sub-minute components | Exclude from the minute-grid experiment | Require exact one-minute horizons |
| Timestamps outside 09:15–15:29 | Exclude from this experiment's defined scope | Keep one consistent intraday analysis window |
| Weekends within the chosen window | Do not exclude solely for day of week | Special sessions cannot be ruled out using weekdays alone |
| Zero volume, positive close at most 1, flat OHLC | Retain and flag | Unusual/illiquid behavior is not automatically erroneous |
| Missing minute history or next-minute target | Exclude the example, retaining otherwise valid bars | Require a well-defined uninterrupted prediction horizon |
| Unverified ninth field and expiry | Exclude from model inputs | Do not invent field meanings or expiry dates |

The selected intraday window is a **modeling scope**, not a claim that every excluded timestamp is erroneous. Vendor timezone/bar conventions remain unverified. Source IDs keep provenance separate across archive files; they are not asserted to be complete exchange contract identifiers. Records sharing time, strike, and call/put across different files are not automatically deduplicated because expiry identity is incomplete.

In [3]:
preparation_counts = pd.DataFrame([
    ('Selected raw rows', summary['selected_raw_rows']),
    ('Clean rows retained', summary['retained_clean_rows']),
    ('Rows excluded by cleaning/scope rules', summary['excluded_rows']),
    ('Eligible supervised examples', summary['total_examples']),
    ('Clean observed dates', summary['observed_dates']),
    ('Model features per example', summary['feature_count']),
], columns=['Metric', 'Count'])
display(preparation_counts)
rules = [name.removeprefix('flag_') for name in summary['counts'] if name.startswith('flag_')]
rule_audit = pd.DataFrame([
    {'Rule': rule, 'Flagged rows (may overlap)': summary['counts'][f'flag_{rule}'],
     'Excluded rows (first applicable reason)': summary['counts'][f'excluded_{rule}']}
    for rule in rules
])
display(rule_audit)
print('Per-archive audit:', REPORT / 'exclusions.csv')
print('Flag counts overlap; first-applicable exclusion counts are mutually exclusive.')
assert summary['selected_raw_rows'] == summary['retained_clean_rows'] + summary['excluded_rows']
assert summary['total_examples'] == int(splits['examples'].sum())
assert int(rule_audit['Excluded rows (first applicable reason)'].sum()) == summary['excluded_rows']

Per-archive audit: /Users/kuijun_2/Desktop/fall-2026-AI-financial-models/reports/nifty_preprocessing/exclusions.csv
Flag counts overlap; first-applicable exclusion counts are mutually exclusive.


,Metric,Count
0,Selected raw rows,17611169
1,Clean rows retained,17576228
2,Rows excluded by cleaning/scope rules,34941
3,Eligible supervised examples,11469619
4,Clean observed dates,1000
5,Model features per example,15


,Rule,Flagged rows (may overlap),Excluded rows (first applicable reason)
0,invalid_timestamp,0,0
1,invalid_symbol_or_strike,0,0
2,missing_or_nonfinite_numeric,0,0
3,nonpositive_price,0,0
4,negative_volume,0,0
5,ohlc_inconsistent,2952,2952
6,conflicting_source_timestamp,0,0
7,exact_duplicate_extra,0,0
8,outside_reference_session,34662,31989
9,non_minute_timestamp,0,0


## 3. Causal feature construction and transformations

For a feature timestamp `t`, the pipeline requires **six consecutive clean bars** from `t−5` through `t`, then an observed target bar exactly one minute later. All seven observations must belong to the same source file and date. Windows never cross missing minutes, days, files, or split boundaries. Neither future prices nor future volume appear in the features.

The regression label is:

$$y_t = \log(C_{t+1}) - \log(C_t).$$

Prices and strike use logarithms, volumes use `log1p`, and OHLC relationships use log ratios. Historical returns, recent return variation, and six-bar mean volume describe the recent path. Sine/cosine time encoding respects cyclic intraday position; call/put and the retained quality flags use binary indicators. The exact feature definitions generated by the pipeline are displayed below.

The first nine continuous features are standardized as `(x − training_mean) / training_std`. **The scaler is fitted on training examples only** and then applied unchanged to validation/test examples. Binary indicators and sine/cosine features stay in their natural scales. The target remains an unscaled log return. No learned imputer is needed for the required fields after cleaning; no gap filling or winsorization is applied.

Fitting transformations only on training data follows the [scikit-learn guidance on avoiding data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

In [4]:
definitions = feature_spec['definitions']
feature_table = pd.DataFrame([
    {'Feature': name, 'Definition': definitions[name],
     'Standardized': name in feature_spec['standardized_columns']}
    for name in feature_columns
])
display(feature_table)
display(pd.DataFrame({
    'Feature': scaler['continuous_columns'],
    'Training mean': scaler['mean'],
    'Training scale': scaler['scale'],
}))
print('Examples used to fit the scaler:', f"{scaler['n_samples_seen']:,}")
assert summary['history_bars'] == 6
assert len(feature_columns) == summary['feature_count']

Examples used to fit the scaler: 7,069,922


,Feature,Definition,Standardized
0,log_close,ln(close[t]),True
1,log_strike,ln(strike),True
2,log_volume,ln(1 + reported volume[t]); units unverified,True
3,log_high_low,"ln(high[t]/low[t]); bar range, not bid-ask spread",True
4,log_close_open,ln(close[t]/open[t]),True
5,return_1m,ln(close[t]/close[t-1]),True
6,return_5m,ln(close[t]/close[t-5]),True
7,return_std_5m,Population standard deviation of five one-minu...,True
8,log_mean_volume_6bars,ln(1 + mean(reported volume[t-5:t])),True
9,minute_sin,sin(2*pi*source minute of day/1440),False


,Feature,Training mean,Training scale
0,log_close,3.932716,1.593188
1,log_strike,9.259285,0.081984
2,log_volume,7.808379,2.528389
3,log_high_low,0.024751,0.119964
4,log_close_open,-0.002565,0.113725
5,return_1m,-0.000320,0.079626
6,return_5m,-0.001583,0.096169
7,return_std_5m,0.019308,0.074614
8,log_mean_volume_6bars,8.158922,2.122573


## 4. Chronological train / validation / test split

The pipeline orders **actual dates with clean observations** and assigns approximately 70% of those dates to training, 15% to validation, and the remaining dates to testing. All source files and strikes on a date stay in the same partition. The fractions refer to dates, not rows: changing market/source coverage means the example percentages can differ substantially.

The date boundaries are fixed before fitting the scaler. Features use only same-day history and targets stay on the same day, so there are no overlapping feature/target windows across partitions. A random row split would make adjacent, strongly related observations appear on both sides of the evaluation boundary.

The earlier EDA already examined the full time period. The test partition is held out from **scaler/model fitting**, but it is not a previously unseen exploration period. Keep it out of further model selection; an additional untouched time period would be needed for a strictly new final evaluation.

In [5]:
split_display = splits.copy()
split_display['date_percent'] = 100 * split_display['dates'] / split_display['dates'].sum()
split_display['example_percent'] = 100 * split_display['examples'] / split_display['examples'].sum()
display(split_display)
display(date_splits.head())
display(date_splits.tail())

ordered = splits.set_index('split').loc[['train', 'valid', 'test']]
assert pd.Timestamp(ordered.loc['train', 'end_date']) < pd.Timestamp(ordered.loc['valid', 'start_date'])
assert pd.Timestamp(ordered.loc['valid', 'end_date']) < pd.Timestamp(ordered.loc['test', 'start_date'])
assert int(scaler['n_samples_seen']) == int(ordered.loc['train', 'examples'])

,split,start_date,end_date,dates,clean_rows,examples,example_percent,date_percent
0,train,2016-12-14,2019-10-17,700,9887527,7069922,61.640426,70.000000
1,valid,2019-10-18,2020-05-29,150,3565817,2009818,17.522971,15.000000
2,test,2020-06-01,2020-12-31,150,4122884,2389879,20.836603,15.000000


,observation_date,split,clean_rows,examples
0,2016-12-14,train,4715,2042
1,2016-12-15,train,5403,3445
2,2016-12-16,train,4941,2318
3,2016-12-19,train,5050,2636
4,2016-12-20,train,5605,3897


,observation_date,split,clean_rows,examples
995,2020-12-24,test,24962,17102
996,2020-12-28,test,25552,17072
997,2020-12-29,test,26309,17527
998,2020-12-30,test,25827,16758
999,2020-12-31,test,24902,14627


## 5. Load model-ready arrays without loading the entire dataset into memory

Each split contains standardized `X.npy`, unscaled `y.npy`, and aligned `metadata.parquet`. The feature order is given by `feature_spec.json`; the fitted preprocessing parameters are in `scaler.json`. `manifest.json` records the preparation settings and input provenance. Clean bars and unscaled eligible examples are also saved in cohort-level Parquet files for auditability.

Memory mapping reads only the slices needed by a training loop. The next cell checks array sizes and reads a small batch; it does not fit a model. Metadata rows are in the same order as the arrays and should be retained for date-level and subgroup evaluation.

In [6]:
arrays = {}
array_rows = []
for split in ('train', 'valid', 'test'):
    X = np.load(DATA / split / 'X.npy', mmap_mode='r')
    y = np.load(DATA / split / 'y.npy', mmap_mode='r')
    metadata_file = pq.ParquetFile(DATA / split / 'metadata.parquet')
    expected = int(ordered.loc[split, 'examples'])
    assert X.shape == (expected, len(feature_columns))
    assert y.shape[0] == expected
    assert metadata_file.metadata.num_rows == expected
    arrays[split] = (X, y)
    array_rows.append({'Split': split, 'X shape': str(X.shape),
                       'y shape': str(y.shape), 'X dtype': str(X.dtype),
                       'Metadata rows': metadata_file.metadata.num_rows})
display(pd.DataFrame(array_rows))

X_train, y_train = arrays['train']
batch_size = min(256, len(y_train))
X_batch = np.array(X_train[:batch_size], copy=True)
y_batch = np.array(y_train[:batch_size], copy=True)
assert np.isfinite(X_batch).all() and np.isfinite(y_batch).all()
display(pd.DataFrame(X_batch[:5], columns=feature_columns))
print('First five labels:', y_batch[:5])
metadata_preview = next(pq.ParquetFile(DATA / 'train/metadata.parquet').iter_batches(batch_size=5))
metadata_preview = metadata_preview.to_pandas()
display(metadata_preview)
assert metadata_preview['label_timestamp'].sub(metadata_preview['timestamp']).eq(pd.Timedelta(minutes=1)).all()
assert metadata_preview['timestamp'].sub(metadata_preview['window_start']).eq(pd.Timedelta(minutes=5)).all()
print('Small batch loaded successfully. No model has been trained.')

First five labels: [-0.01186702 -0.00624785  0.          0.00084926  0.00508045]
Small batch loaded successfully. No model has been trained.


,Split,X shape,y shape,X dtype,Metadata rows
0,train,"(7069922, 15)","(7069922,)",float32,7069922
1,valid,"(2009818, 15)","(2009818,)",float32,2009818
2,test,"(2389879, 15)","(2389879,)",float32,2389879


,log_close,log_strike,log_volume,log_high_low,log_close_open,return_1m,return_5m,return_std_5m,log_mean_volume_6bars,minute_sin,minute_cos,is_call,zero_volume,close_at_most_1,flat_ohlc
0,1.095651,-3.472227,-1.103904,-0.203467,0.019548,0.010457,0.030689,-0.184880,-1.240697,0.625923,-0.779884,1.000000,0.000000,0.000000,0.000000
1,1.088202,-3.472227,-1.103904,-0.206317,0.022554,-0.145020,-0.037374,-0.162218,-1.290126,0.622515,-0.782608,1.000000,0.000000,0.000000,1.000000
2,1.099502,-3.472227,-0.365812,-0.163931,-0.020670,-0.068334,-0.043445,-0.227073,-0.390336,-0.026177,-0.999657,1.000000,0.000000,0.000000,0.000000
3,1.099502,-3.472227,-0.743078,-0.165341,-0.020670,0.004014,-0.036416,-0.226506,-0.474630,-0.030539,-0.999534,1.000000,0.000000,0.000000,0.000000
4,1.100036,-3.472227,-0.259807,-0.155514,-0.029543,0.014680,-0.022309,-0.224954,-0.474630,-0.034899,-0.999391,1.000000,0.000000,0.000000,0.000000


,file_id,source_row,window_start,timestamp,label_timestamp,option_type,strike,close,target_close,archive_cohort,split
0,120,180,2016-12-21 09:20:00,2016-12-21 09:25:00,2016-12-21 09:26:00,CE,"7,900.000000",292.450000,289.000000,2017-01,train
1,120,181,2016-12-21 09:21:00,2016-12-21 09:26:00,2016-12-21 09:27:00,CE,"7,900.000000",289.000000,287.200000,2017-01,train
2,120,230,2016-12-21 12:01:00,2016-12-21 12:06:00,2016-12-21 12:07:00,CE,"7,900.000000",294.250000,294.250000,2017-01,train
3,120,231,2016-12-21 12:02:00,2016-12-21 12:07:00,2016-12-21 12:08:00,CE,"7,900.000000",294.250000,294.500000,2017-01,train
4,120,232,2016-12-21 12:03:00,2016-12-21 12:08:00,2016-12-21 12:09:00,CE,"7,900.000000",294.500000,296.000000,2017-01,train


## 6. Is there enough data for deep learning?

Millions of neighboring minute-level records do not represent millions of independent market conditions. Consecutive windows overlap heavily; option strikes and calls/puts on the same date share market shocks. The number of observed dates and the coverage of different market regimes matter alongside the raw example count. The historical period also has changes in source coverage and schema.

The prepared dataset is large enough to attempt a compact model, but sample size alone cannot guarantee that deep learning will avoid overfitting. Begin with a **zero-return baseline** and a regularized linear model, then compare a small neural network using validation-only early stopping and weight decay. Track training versus validation loss, evaluate error across dates and quality-flag groups, and use date-blocked uncertainty estimates. Any additional feature engineering or learned transformations must be fitted on the training partition. Select the model using validation data; reserve the test partition for a single final comparison once choices are fixed.

For general methods to control neural-network capacity and overfitting, see the [TensorFlow tutorial on overfitting and underfitting](https://www.tensorflow.org/tutorials/keras/overfit_and_underfit). Dataset-specific counts and conclusions above come from this repository's audit outputs.

These are modeling recommendations for the next stage. This notebook does not claim a trained model, validation score, trading strategy, or demonstrated predictive skill.

## 7. Copy-ready assignment answers

The following text is generated from the actual saved preparation reports. Copy the rendered output, including the distinction between the chosen forecasting target and a theoretical option-pricing model.

In [7]:
split_lines = []
for split, label in [('train', 'Training'), ('valid', 'Validation'), ('test', 'Test')]:
    row = ordered.loc[split]
    split_lines.append(
        f"- **{label}:** {int(row['examples']):,} examples across {int(row['dates']):,} dates "
        f"({row['start_date']} to {row['end_date']}); "
        f"{100 * row['examples'] / summary['total_examples']:.1f}% of examples."
    )
answer = f"""
**Dataset and basic EDA.** The Nifty options dataset is already available in the workspace. It contains {eda['selected_rows']:,} selected observations from {eda['selected_files']:,} source files after excluding identical CSV/TXT representations. The observations span {eda['first_timestamp'][:10]} to {eda['last_timestamp'][:10]}. EDA examined OHLC prices, volume, call/put categories, timestamp coverage, missing fields, duplicates, and intraday gaps. Close prices have median {close_stats['median']:,.2f} and mean {close_stats['mean']:,.2f}; reported volume has median {volume_stats['median']:,.0f} and mean {volume_stats['mean']:,.2f}. The long upper tails and means above medians indicate right-skewed distributions, motivating logarithmic transformations. The audit found {eda['quality_counts']['ohlc_inconsistent']:,} inconsistent OHLC rows, {eda['quality_counts']['zero_volume']:,} zero-volume rows, and an unverified ninth field missing from {100 * eda['quality_counts']['missing_extra_field_9'] / eda['selected_rows']:.1f}% of records. Minute coverage is incomplete, so valid prediction windows must be identified explicitly.

**1. Should all data be included, or are inclusion/exclusion criteria needed?** Inclusion/exclusion criteria are needed. I removed duplicate file representations and excluded invalid required values, invalid OHLC/volume rows, and observations outside the experiment's 09:15–15:29 intraday window. This window defines the experiment's scope; it is not proof that outside-window data are erroneous, and the vendor timezone remains unverified. The preparation retained {summary['retained_clean_rows']:,} clean rows and excluded {summary['excluded_rows']:,} rows from the selected observations. I retained zero-volume, small positive-price, and flat-price records with indicator features, because they are not automatically data errors. The unverified ninth field and incomplete expiry information are excluded from model inputs. Each supervised example requires six consecutive clean bars in one source file on one date plus an observed next-minute target; missing minutes are not filled.

**2. What transformations/preprocessing are needed before modeling?** I parsed dates, times, symbols, and numeric fields; kept source provenance; applied the cleaning rules; and built {summary['feature_count']} causal features. These include log price/strike, log1p volume, OHLC log ratios, historical returns and return variation, recent average volume, cyclical time features, call/put, and quality indicators. Continuous features are standardized using training-only means and standard deviations, then the same transformation is applied to validation and test data. The chosen regression target is the next-minute option log price change, log(close[t+1]/close[t]), which remains unscaled. Multiplying the current close by exp(predicted log return) converts a prediction to the next option close. The outputs are aligned feature arrays, target arrays, metadata, and saved preprocessing parameters. They are ready for this forecasting experiment; a theoretical option-pricing model still requires verified contract/expiry details and synchronized underlying-price and other pricing inputs.

**3. How much data goes into training/validation/testing, and will deep learning overfit?** I used a chronological split of approximately 70%/15%/15% of clean observed dates, keeping every observation from a date in one partition. Because coverage varies, the example percentages differ from the date percentages:

{chr(10).join(split_lines)}

There are {summary['total_examples']:,} eligible supervised examples in total. This supports trying a small neural network, but it does not guarantee freedom from overfitting: neighboring windows overlap and many options share the same market conditions. I would compare against zero-return and regularized linear baselines, use a compact network with weight decay and validation-based early stopping, inspect train/validation losses and date/subgroup errors, and keep the test set out of model selection. The previous EDA already examined all periods, so the test set is held out from fitting but is not a previously unseen exploration period. A separate untouched period is needed for a strictly new final evaluation.

Method references: [scikit-learn: avoiding data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage); [TensorFlow: overfitting and underfitting](https://www.tensorflow.org/tutorials/keras/overfit_and_underfit). Dataset counts are from the local EDA and preprocessing reports.
"""
display(Markdown(answer))


**Dataset and basic EDA.** The Nifty options dataset is already available in the workspace. It contains 17,611,169 selected observations from 3,137 source files after excluding identical CSV/TXT representations. The observations span 2016-12-14 to 2020-12-31. EDA examined OHLC prices, volume, call/put categories, timestamp coverage, missing fields, duplicates, and intraday gaps. Close prices have median 88.00 and mean 171.71; reported volume has median 1,125 and mean 9,228.15. The long upper tails and means above medians indicate right-skewed distributions, motivating logarithmic transformations. The audit found 2,952 inconsistent OHLC rows, 396,423 zero-volume rows, and an unverified ninth field missing from 85.1% of records. Minute coverage is incomplete, so valid prediction windows must be identified explicitly.

**1. Should all data be included, or are inclusion/exclusion criteria needed?** Inclusion/exclusion criteria are needed. I removed duplicate file representations and excluded invalid required values, invalid OHLC/volume rows, and observations outside the experiment's 09:15–15:29 intraday window. This window defines the experiment's scope; it is not proof that outside-window data are erroneous, and the vendor timezone remains unverified. The preparation retained 17,576,228 clean rows and excluded 34,941 rows from the selected observations. I retained zero-volume, small positive-price, and flat-price records with indicator features, because they are not automatically data errors. The unverified ninth field and incomplete expiry information are excluded from model inputs. Each supervised example requires six consecutive clean bars in one source file on one date plus an observed next-minute target; missing minutes are not filled.

**2. What transformations/preprocessing are needed before modeling?** I parsed dates, times, symbols, and numeric fields; kept source provenance; applied the cleaning rules; and built 15 causal features. These include log price/strike, log1p volume, OHLC log ratios, historical returns and return variation, recent average volume, cyclical time features, call/put, and quality indicators. Continuous features are standardized using training-only means and standard deviations, then the same transformation is applied to validation and test data. The chosen regression target is the next-minute option log price change, log(close[t+1]/close[t]), which remains unscaled. Multiplying the current close by exp(predicted log return) converts a prediction to the next option close. The outputs are aligned feature arrays, target arrays, metadata, and saved preprocessing parameters. They are ready for this forecasting experiment; a theoretical option-pricing model still requires verified contract/expiry details and synchronized underlying-price and other pricing inputs.

**3. How much data goes into training/validation/testing, and will deep learning overfit?** I used a chronological split of approximately 70%/15%/15% of clean observed dates, keeping every observation from a date in one partition. Because coverage varies, the example percentages differ from the date percentages:

- **Training:** 7,069,922 examples across 700 dates (2016-12-14 to 2019-10-17); 61.6% of examples.
- **Validation:** 2,009,818 examples across 150 dates (2019-10-18 to 2020-05-29); 17.5% of examples.
- **Test:** 2,389,879 examples across 150 dates (2020-06-01 to 2020-12-31); 20.8% of examples.

There are 11,469,619 eligible supervised examples in total. This supports trying a small neural network, but it does not guarantee freedom from overfitting: neighboring windows overlap and many options share the same market conditions. I would compare against zero-return and regularized linear baselines, use a compact network with weight decay and validation-based early stopping, inspect train/validation losses and date/subgroup errors, and keep the test set out of model selection. The previous EDA already examined all periods, so the test set is held out from fitting but is not a previously unseen exploration period. A separate untouched period is needed for a strictly new final evaluation.

Method references: [scikit-learn: avoiding data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage); [TensorFlow: overfitting and underfitting](https://www.tensorflow.org/tutorials/keras/overfit_and_underfit). Dataset counts are from the local EDA and preprocessing reports.


## Reproducibility and outputs

- Run `python scripts/nifty_preprocess.py` to build the preparation outputs, then rerun this notebook to refresh displayed results.
- `reports/nifty_preprocessing/summary.json`, `split_summary.csv`, `exclusions.csv`, and `date_splits.csv` preserve the summary counts, exclusions, and split boundaries.
- `data/processed/nifty_training/` contains the local clean bars, examples, memory-mapped arrays, metadata, feature specification, scaler, and manifest. Bulk data are Git-ignored.
- `scripts/build_nifty_preprocessing_notebook.py` regenerates this notebook's source cells. Regeneration clears executed outputs; run the notebook afterward to restore them.
- `reports/nifty_preprocessing/preprocessing.html` is the rendered walkthrough from the executed notebook.

Cached results describe the inputs used when the pipeline ran. After changing raw files, rebuild the inventory/EDA and preprocessing so provenance and statistics remain aligned. Keep this preprocessing configuration fixed while comparing models; changing the target or inclusion criteria creates a new experiment that requires new arrays and a fresh audit.